In [1]:
# Phase 6.6 — Export all four trained PyTorch checkpoints to full-precision ONNX
# Quantization is intentionally skipped: INT8 dynamic quantization breaks LSTM/transformer
# accuracy (video F1 dropped from 0.87 to 0.07). Full-precision ONNX is fast enough for
# the project's <10s CPU inference budget.
from __future__ import annotations

import argparse
import time
from pathlib import Path

WORKING_DIR = Path("/kaggle/working")

_CKPT_NAMES = {
    "video":  "efficientnet_lstm.pt",
    "audio":  "wav2vec2_classifier.pt",
    "text":   "xlm_roberta_sentiment.pt",
    "fusion": "cross_modal_transformer.pt",
}


# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------

def export_run(model: str = "all", skip_export: bool = False) -> None:
    """Notebook entry point — call this directly (e.g. `export_run()` or
    `export_run("audio")`) instead of main(). Running the whole script in a notebook cell
    triggers `if __name__ == "__main__"`, and argparse then reads the kernel's own
    sys.argv (e.g. Colab/Kaggle's `-f kernel.json` launcher flag) instead of your intended
    arguments, raising 'unrecognized arguments'. This function takes plain parameters
    instead, so it has nothing to do with sys.argv. One call now handles all four models
    in a single run and skips any model whose checkpoint isn't available yet, rather than
    stopping partway. Named export_run (not run) because evaluate_models.py also defines
    a notebook entry point called run() — both pasted into the same notebook would
    silently overwrite each other under the same name."""
    t_total = time.time()
    print(f"[Export] Target: {model}")

    if model in ("all", "video"):
        _handle("video", skip_export)
    if model in ("all", "audio"):
        _handle("audio", skip_export)
    if model in ("all", "text"):
        _handle("text", skip_export)
    if model in ("all", "fusion"):
        _handle("fusion", skip_export)

    elapsed = (time.time() - t_total) / 60
    print(f"\n[Export] All done in {elapsed:.1f} min.")
    _print_copy_instructions()


def main() -> None:
    parser = argparse.ArgumentParser(description="Export + quantize DeepCue models.")
    parser.add_argument("--model", default="all",
                        choices=["all", "video", "audio", "text", "fusion"])
    parser.add_argument("--skip_export", action="store_true",
                        help="Skip PyTorch → ONNX step; only quantize existing .onnx files.")
    args = parser.parse_args()
    export_run(args.model, args.skip_export)


def _handle(model_name: str, skip_export: bool) -> None:
    onnx_path = _onnx_path(model_name)

    if not skip_export:
        ckpt_path = WORKING_DIR / _CKPT_NAMES[model_name]
        if not ckpt_path.exists():
            print(f"[Export] {model_name}: checkpoint {ckpt_path.name} not found — skipping entirely.")
            return
        t0 = time.time()
        print(f"\n[Export] {model_name}: PyTorch → ONNX ...")
        _export(model_name)
        print(f"[Export] {model_name}: ONNX written in {time.time()-t0:.1f}s")

    if not onnx_path.exists():
        print(f"[Export] {model_name}: {onnx_path} not found.")
        return

    size_mb = onnx_path.stat().st_size / 1024 / 1024
    print(f"[Export] {model_name}: {onnx_path.name}  ({size_mb:.1f} MB)")


def _export(model_name: str) -> None:
    """Call the training script's export_onnx() to re-use its model definition."""
    if model_name == "video":
        from train_video_model import export_onnx
        export_onnx()
    elif model_name == "audio":
        from train_audio_model import export_onnx
        export_onnx()
    elif model_name == "text":
        from finetune_xlm_roberta import export_onnx
        export_onnx()
    elif model_name == "fusion":
        from train_fusion_model import export_onnx
        export_onnx()


def _onnx_path(model_name: str) -> Path:
    names = {
        "video":  "efficientnet_lstm.onnx",
        "audio":  "wav2vec2_classifier.onnx",
        "text":   "xlm_roberta_sentiment.onnx",
        "fusion": "cross_modal_transformer.onnx",
    }
    return WORKING_DIR / names[model_name]


def _print_copy_instructions() -> None:
    print("\n" + "=" * 60)
    print("Download from Kaggle output panel, then copy to backend:")
    print()
    print("  efficientnet_lstm.onnx        → models/video/efficientnet_lstm.onnx")
    print("  wav2vec2_classifier.onnx      → models/audio/wav2vec2_classifier.onnx")
    print("  xlm_roberta_sentiment.onnx    → models/text/xlm_roberta_sentiment.onnx")
    print("  cross_modal_transformer.onnx  → models/fusion/cross_modal_transformer.onnx")
    print()
    print("Then update VIDEO/AUDIO/TEXT/FUSION_MODEL_PATH in your .env file.")
    print("=" * 60)



In [3]:
!pip install onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 75.8 MB/s eta 0:00:00:00:0100:01


In [8]:
# Phase 6.5 — Evaluate all four quantized ONNX models; each must reach Macro F1 >= 0.50 (item 8.5)
from __future__ import annotations

import argparse
import random
import time
from pathlib import Path

import librosa
import numpy as np
import onnxruntime as ort
from sklearn.metrics import classification_report, f1_score

WORKING_DIR  = Path("/kaggle/working")
# Full RAVDESS (MP4 + WAV) — attach dataset orvile/ravdess-dataset
RAVDESS_ROOT = Path("/kaggle/input/datasets/orvile/ravdess-dataset")
HEBREW_SENTIMENT_HF_DATASET = "omilab/hebrew_sentiment"
SAMPLE_RATE  = 16000
AUDIO_LEN    = 48000
IMAGE_SIZE   = 224
MAX_LEN      = 128
WINDOW_SIZE  = 8
SEED         = 42
F1_THRESHOLD = 0.50

EMOTION_CLASSES = [
    "neutral", "confident", "anxious", "happy",
    "sad", "angry", "surprised", "uncertain",
]

# RAVDESS filename emotion code (field 3) → DeepCue class
_RAVDESS_MAP: dict[int, int] = {
    1: 0, 2: 0,  # neutral + calm → neutral
    3: 3,        # happy
    4: 4,        # sad
    5: 5,        # angry
    6: 2,        # fearful → anxious
    7: 7,        # disgust → uncertain
    8: 6,        # surprised
}


# ---------------------------------------------------------------------------
# Shared data helpers
# ---------------------------------------------------------------------------

def _subsample(items: list, max_samples: int | None) -> list:
    """Randomly subsample (fixed seed, for reproducibility) when max_samples is given —
    lets you sanity-check a model in minutes instead of running the full RAVDESS/HF set,
    which can take hours. Use the full (max_samples=None) run only for the official
    F1 >= 0.50 gate-check, not for quick debugging."""
    if max_samples is None or max_samples >= len(items):
        return items
    return random.Random(SEED).sample(items, max_samples)


def _ravdess_samples(root: Path) -> list[tuple[Path, int]]:
    """Walk RAVDESS root and collect (mp4_path, label) pairs."""
    samples = []
    for mp4 in sorted(root.rglob("*.mp4")):
        parts = mp4.stem.split("-")
        if len(parts) >= 3:
            try:
                label = _RAVDESS_MAP.get(int(parts[2]))
                if label is not None:
                    samples.append((mp4, label))
            except ValueError:
                pass
    return samples


def _load_audio(path: Path) -> np.ndarray:
    """Load MP4 audio stream, resample to 16 kHz, pad/truncate to 3 s."""
    import torchaudio
    waveform, sr = torchaudio.load(str(path))
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)  # mono
    if sr != SAMPLE_RATE:
        import torchaudio.transforms as T
        waveform = T.Resample(sr, SAMPLE_RATE)(waveform)
    audio = waveform.squeeze().numpy()
    audio = (audio - audio.mean()) / np.sqrt(audio.var() + 1e-7)  # match train_audio_model.py
    if len(audio) < AUDIO_LEN:
        audio = np.pad(audio, (0, AUDIO_LEN - len(audio)))
    return audio[:AUDIO_LEN].astype(np.float32)


def _extract_audio_features(waveform: np.ndarray) -> np.ndarray:
    """16-dim feature vector matching AudioEmotionPipeline: pitch, RMS, ZCR, 13 MFCCs."""
    f0     = librosa.yin(waveform, fmin=80.0, fmax=400.0, sr=SAMPLE_RATE)
    voiced = f0[f0 > 0]
    pitch  = float(np.mean(voiced) / 400.0) if len(voiced) > 0 else 0.0
    rms    = float(np.mean(librosa.feature.rms(y=waveform)[0]))
    zcr    = float(np.mean(librosa.feature.zero_crossing_rate(waveform)[0]))
    mfccs  = librosa.feature.mfcc(y=waveform, sr=SAMPLE_RATE, n_mfcc=13).mean(axis=1).tolist()
    return np.array([pitch, rms, zcr] + mfccs, dtype=np.float32)


def _load_frames(path: Path, n_frames: int = WINDOW_SIZE) -> np.ndarray:
    """Sample n_frames evenly across a video; returns [n_frames, 3, H, W] float32.
    Seeks directly to each target frame index instead of decoding every frame in between
    — a RAVDESS clip has ~100+ frames at native framerate, so decoding all of them just to
    keep 8 made this the dominant cost of evaluation (~1.5s/sample, hours for the full set)."""
    import cv2
    cap   = cv2.VideoCapture(str(path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 1
    idxs  = np.linspace(0, total - 1, min(n_frames, total), dtype=int)
    frames: list[np.ndarray] = []
    for idx in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if not ret:
            continue
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, (IMAGE_SIZE, IMAGE_SIZE))
        frames.append(frame.transpose(2, 0, 1).astype(np.float32) / 255.0)
    cap.release()
    if not frames:
        frames = [np.zeros((3, IMAGE_SIZE, IMAGE_SIZE), dtype=np.float32)]
    arr = np.stack(frames)
    if len(arr) < n_frames:
        pad = np.tile(arr[-1:], (n_frames - len(arr), 1, 1, 1))
        arr = np.concatenate([arr, pad], axis=0)
    return arr[:n_frames]


# ---------------------------------------------------------------------------
# Per-model evaluators
# ---------------------------------------------------------------------------

def evaluate_video(onnx_path: Path, max_samples: int | None = None) -> float:
    print(f"\n[Video] Loading {onnx_path.name}")
    sess       = ort.InferenceSession(str(onnx_path), providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
    input_name = sess.get_inputs()[0].name
    samples    = _ravdess_samples(RAVDESS_ROOT)
    samples    = _subsample(samples, max_samples)
    print(f"[Video] {len(samples)} RAVDESS samples")
    preds, labels = [], []
    t0 = time.time()

    for i, (mp4, label) in enumerate(samples):
        frames  = _load_frames(mp4, WINDOW_SIZE)
        seq     = frames[np.newaxis].astype(np.float32)  # [1, W, 3, H, H]
        out     = sess.run(None, {input_name: seq})[0].flatten()
        pred    = int(np.argmax(out)) if len(out) == 8 else _score_to_class(out[0])
        preds.append(pred)
        labels.append(label)
        if (i + 1) % 100 == 0:
            print(f"  [Video] {i+1}/{len(samples)}  ({time.time()-t0:.0f}s elapsed)")

    return _report("Video", labels, preds)


def evaluate_audio(onnx_path: Path, max_samples: int | None = None) -> float:
    print(f"\n[Audio] Loading {onnx_path.name}")
    sess    = ort.InferenceSession(str(onnx_path), providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
    samples = _ravdess_samples(RAVDESS_ROOT)
    samples = _subsample(samples, max_samples)
    print(f"[Audio] {len(samples)} RAVDESS samples")
    preds, labels = [], []
    t0 = time.time()

    for i, (mp4, label) in enumerate(samples):
        audio = _load_audio(mp4)
        feats = _extract_audio_features(audio)
        out   = sess.run(None, {
            "audio_waveform": audio[np.newaxis],
            "features":       feats[np.newaxis],
        })[0].flatten()
        preds.append(int(np.argmax(out)))  # audio now exports 8-class logits, not a scalar
        labels.append(label)
        if (i + 1) % 100 == 0:
            print(f"  [Audio] {i+1}/{len(samples)}  ({time.time()-t0:.0f}s elapsed)")

    return _report("Audio", labels, preds)


# omilab/hebrew_sentiment label int → scalar score in [0, 1] — must match the mapping used
# in finetune_xlm_roberta.py's training data, so evaluation classes are computed consistently.
_HEBREW_SENTIMENT_LABEL_SCORE: dict[int, float] = {0: 1.0, 1: 0.0, 2: 0.5}


def evaluate_text(onnx_path: Path, max_samples: int | None = None) -> float:
    """Evaluate text model by MAE on its raw [0,1] scalar output — not by 8-class F1.
    The text model is a sentiment regressor, not an emotion classifier; forcing it into
    the 8-class F1 gate produces meaningless results (only 3 of 8 classes ever appear,
    macro F1 averages over all 8). MAE directly measures regression quality."""
    print(f"\n[Text] Loading {onnx_path.name}")
    from datasets import load_dataset
    from sklearn.metrics import mean_absolute_error
    from transformers import AutoTokenizer

    sess      = ort.InferenceSession(str(onnx_path), providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
    tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
    hf_dataset = load_dataset(HEBREW_SENTIMENT_HF_DATASET, revision="refs/convert/parquet")["test"]
    pairs      = _subsample(list(zip(hf_dataset["text"], hf_dataset["label"])), max_samples)
    texts      = [t for t, _ in pairs]
    gt_scores  = [_HEBREW_SENTIMENT_LABEL_SCORE[label] for _, label in pairs]
    pred_scores: list[float] = []
    t0 = time.time()
    print(f"[Text] {len(texts)} {HEBREW_SENTIMENT_HF_DATASET} (test) samples")

    for i, text in enumerate(texts):
        enc = tokenizer(text, return_tensors="np", max_length=MAX_LEN,
                        truncation=True, padding="max_length")
        out = sess.run(None, {
            "input_ids":      enc["input_ids"].astype(np.int64),
            "attention_mask": enc["attention_mask"].astype(np.int64),
        })[0].flatten()
        pred_scores.append(float(out[0]))
        if (i + 1) % 200 == 0:
            print(f"  [Text] {i+1}/{len(texts)}  ({time.time()-t0:.0f}s elapsed)")

    mae = float(mean_absolute_error(gt_scores, pred_scores))
    print(f"\n{'='*50}")
    print(f"  Text — MAE: {mae:.4f}  (lower is better; <0.15 = PASS)")
    status = "PASS" if mae < 0.15 else "BELOW THRESHOLD"
    print(f"  [{status}]  MAE={mae:.4f}")
    return mae


def evaluate_fusion(onnx_path: Path) -> float:
    print(f"\n[Fusion] Loading {onnx_path.name}")
    sess       = ort.InferenceSession(str(onnx_path), providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
    input_name = sess.get_inputs()[0].name

    # Synthetic 17-dim test data matching train_fusion_model.py's _build_synthetic_dataset:
    # video[8] + audio[8] logit vectors peaked at the true class, text[1] scalar score.
    rng = np.random.default_rng(42)
    # Must match train_fusion_model.py's _SYNTHETIC_MEANS exactly (vm, am, tm) and same
    # bump formula so the test distribution matches what the model was trained on.
    _MEANS = {
        0: (0.50, 0.50, 0.52), 1: (0.85, 0.80, 0.80), 2: (0.20, 0.80, 0.32),
        3: (0.90, 0.85, 0.92), 4: (0.15, 0.20, 0.22), 5: (0.90, 0.90, 0.12),
        6: (0.70, 0.95, 0.62), 7: (0.40, 0.30, 0.42),
    }
    X_list, y = [], []
    for label, (vm, am, tm) in _MEANS.items():
        n = 50
        v = rng.normal(0, 1.0, (n, 8))
        v[:, label] += 1.5 + vm * 3.0
        a = rng.normal(0, 1.0, (n, 8))
        a[:, label] += 1.5 + am * 3.0
        t = np.clip(rng.normal(tm, 0.08, (n, 1)), 0, 1)
        X_list.append(np.concatenate([v, a, t], axis=1))
        y.extend([label] * n)
    X_arr = np.vstack(X_list).astype(np.float32)
    print(f"[Fusion] {len(X_arr)} synthetic samples")

    preds = []
    for row in X_arr:
        out = sess.run(None, {input_name: row[np.newaxis]})[0].flatten()
        preds.append(int(np.argmax(out)))

    return _report("Fusion", y, preds)


# ---------------------------------------------------------------------------
# Shared reporting
# ---------------------------------------------------------------------------

def _score_to_class(score: float) -> int:
    """Map [0,1] scalar → 8-class index (equal-width buckets). Used for audio/text outputs."""
    return min(int(score * 8), 7)


def _report(name: str, labels: list[int], preds: list[int]) -> float:
    all_classes = list(range(len(EMOTION_CLASSES)))
    f1 = float(f1_score(labels, preds, average="macro", labels=all_classes, zero_division=0))
    print(f"\n{'='*50}")
    print(f"  {name} — Macro F1: {f1:.4f}  (threshold: {F1_THRESHOLD})")
    # labels=all_classes: a small max_samples subset may not contain every one of the 8
    # classes, which otherwise crashes classification_report with a class-count mismatch.
    print(classification_report(labels, preds, labels=all_classes, target_names=EMOTION_CLASSES, zero_division=0))
    status = "PASS" if f1 >= F1_THRESHOLD else "BELOW THRESHOLD"
    print(f"  [{status}]  {f1:.4f}")
    return f1


# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------

def run(
    model: str = "all",
    video_onnx: Path | None = None,
    audio_onnx: Path | None = None,
    text_onnx: Path | None = None,
    fusion_onnx: Path | None = None,
    max_samples: int | None = None,
) -> None:
    """Notebook entry point — call this directly (e.g. `run()` or `run("audio")`) instead
    of main(). Running the whole script in a notebook cell triggers `if __name__ ==
    "__main__"`, and argparse then reads the kernel's own sys.argv (e.g. Colab/Kaggle's
    `-f kernel.json` launcher flag) instead of your intended arguments, raising
    'unrecognized arguments'. This function takes plain parameters instead.

    max_samples: cap on how many RAVDESS/HF samples to evaluate per model, e.g.
    run("video", max_samples=300) for a quick sanity check in minutes instead of the
    multi-hour full-dataset pass. Leave as None only for the official F1 >= 0.50 gate-check."""
    # Defaults point to full-precision ONNX — quantization is skipped for this project
    # because INT8 dynamic quantization breaks LSTM/transformer models (video F1 0.87→0.07).
    video_onnx  = video_onnx  or (WORKING_DIR / "efficientnet_lstm.onnx")
    audio_onnx  = audio_onnx  or (WORKING_DIR / "wav2vec2_classifier.onnx")
    text_onnx   = text_onnx   or (WORKING_DIR / "xlm_roberta_sentiment.onnx")
    fusion_onnx = fusion_onnx or (WORKING_DIR / "cross_modal_transformer.onnx")
    t_start = time.time()

    f1_results:  dict[str, float] = {}
    mae_results: dict[str, float] = {}

    if model in ("all", "video"):
        f1_results["video"]  = evaluate_video(Path(video_onnx), max_samples)
    if model in ("all", "audio"):
        f1_results["audio"]  = evaluate_audio(Path(audio_onnx), max_samples)
    if model in ("all", "text"):
        # Text is a regression model — evaluated by MAE, not F1; excluded from the F1 gate.
        mae_results["text"]  = evaluate_text(Path(text_onnx), max_samples)
    if model in ("all", "fusion"):
        f1_results["fusion"] = evaluate_fusion(Path(fusion_onnx))

    elapsed  = (time.time() - t_start) / 60
    all_pass = all(f1 >= F1_THRESHOLD for f1 in f1_results.values() if not np.isnan(f1))

    print(f"\n{'='*50}  EVALUATION SUMMARY  {'='*50}")
    for name, f1 in f1_results.items():
        status = "PASS" if f1 >= F1_THRESHOLD else "FAIL"
        print(f"  {name:8s}  F1={f1:.4f}  [{status}]")
    for name, mae in mae_results.items():
        status = "PASS" if mae < 0.15 else "FAIL"
        print(f"  {name:8s}  MAE={mae:.4f}  [{status}]  (regression — lower is better)")
    print(f"\nTotal evaluation time: {elapsed:.1f} min")
    if all_pass:
        print("All models meet their threshold. Ready for deployment.")
    else:
        print("One or more models are below threshold. Review training and data quality.")


def main() -> None:
    parser = argparse.ArgumentParser(description="Evaluate DeepCue ONNX models.")
    parser.add_argument("--model", default="all",
                        choices=["all", "video", "audio", "text", "fusion"])
    parser.add_argument("--video_onnx",  default=str(WORKING_DIR / "efficientnet_lstm_quant.onnx"))
    parser.add_argument("--audio_onnx",  default=str(WORKING_DIR / "wav2vec2_classifier_quant.onnx"))
    parser.add_argument("--text_onnx",   default=str(WORKING_DIR / "xlm_roberta_sentiment_quant.onnx"))
    parser.add_argument("--fusion_onnx", default=str(WORKING_DIR / "cross_modal_transformer_quant.onnx"))
    args = parser.parse_args()
    run(args.model, Path(args.video_onnx), Path(args.audio_onnx), Path(args.text_onnx), Path(args.fusion_onnx))


In [11]:
import shutil, os

src_pt = "/kaggle/input/datasets/rachelsade/pt-files"
src_py = "/kaggle/input/datasets/rachelsade/training-python-file"

for f in ["efficientnet_lstm.pt", "wav2vec2_classifier.pt",
          "xlm_roberta_sentiment.pt", "cross_modal_transformer.pt"]:
    shutil.copy(f"{src_pt}/{f}", f"/kaggle/working/{f}")

for f in ["train_video_model.py", "train_audio_model.py",
          "finetune_xlm_roberta.py", "train_fusion_model.py"]:
    shutil.copy(f"{src_py}/{f}", f"/kaggle/working/{f}")

print("Done:", os.listdir("/kaggle/working"))

Done: ['.virtual_documents', 'cross_modal_transformer.pt', 'train_video_model.py', 'xlm_roberta_sentiment.pt', 'efficientnet_lstm.pt', 'wav2vec2_classifier.pt', 'train_fusion_model.py', 'train_audio_model.py', 'finetune_xlm_roberta.py']


In [12]:
export_run()	

[Export] Target: all

[Export] video: PyTorch → ONNX ...


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset9.py:4445: UserWarning: Exporting a model to ONNX with a batch_size other than 1, with a variable length with LSTM can cause an error when running the ONNX model with a different batch size. Make sure to save the model with a batch size of 1, or define the initial states (h0/c0) as inputs of the model. 
  return _generic_rnn(


ONNX model exported: /kaggle/working/efficientnet_lstm.onnx
[Export] video: ONNX written in 25.0s
[Export] video: efficientnet_lstm.onnx  (33.3 MB)

[Export] audio: PyTorch → ONNX ...

[Audio] Exporting to ONNX ...


config.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/transformers/masking_utils.py:171: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if (padding_length := kv_length + kv_offset - attention_mask.shape[-1]) > 0:
/usr/local/lib/python3.12/dist-packages/transformers/integrations/sdpa_attention.py:77: Tracer

[Text] ONNX exported: /kaggle/working/xlm_roberta_sentiment.onnx
[Export] text: ONNX written in 20.9s
[Export] text: xlm_roberta_sentiment.onnx  (1058.7 MB)

[Export] fusion: PyTorch → ONNX ...

[Fusion] Exporting to ONNX ...


/kaggle/working/train_fusion_model.py:146: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)


[Fusion] ONNX exported: /kaggle/working/cross_modal_transformer.onnx
[Export] fusion: ONNX written in 0.3s
[Export] fusion: cross_modal_transformer.onnx  (1.1 MB)

[Export] All done in 1.3 min.

Download from Kaggle output panel, then copy to backend:

  efficientnet_lstm.onnx        → models/video/efficientnet_lstm.onnx
  wav2vec2_classifier.onnx      → models/audio/wav2vec2_classifier.onnx
  xlm_roberta_sentiment.onnx    → models/text/xlm_roberta_sentiment.onnx
  cross_modal_transformer.onnx  → models/fusion/cross_modal_transformer.onnx

Then update VIDEO/AUDIO/TEXT/FUSION_MODEL_PATH in your .env file.


In [13]:
run(max_samples=300)


[Video] Loading efficientnet_lstm.onnx


/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:147: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


[Video] 300 RAVDESS samples
  [Video] 100/300  (53s elapsed)
  [Video] 200/300  (109s elapsed)
  [Video] 300/300  (164s elapsed)

  Video — Macro F1: 0.7984  (threshold: 0.5)
              precision    recall  f1-score   support

     neutral       0.91      0.98      0.94        59
   confident       0.00      0.00      0.00         0
     anxious       0.98      0.87      0.92        55
       happy       0.92      1.00      0.96        59
         sad       1.00      0.95      0.97        41
       angry       0.81      0.88      0.84        43
   surprised       0.94      0.68      0.79        22
   uncertain       0.95      0.95      0.95        21

    accuracy                           0.92       300
   macro avg       0.81      0.79      0.80       300
weighted avg       0.93      0.92      0.92       300

  [PASS]  0.7984

[Audio] Loading wav2vec2_classifier.onnx


/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:147: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


[Audio] 300 RAVDESS samples
  [Audio] 100/300  (59s elapsed)
  [Audio] 200/300  (102s elapsed)
  [Audio] 300/300  (145s elapsed)

  Audio — Macro F1: 0.4467  (threshold: 0.5)
              precision    recall  f1-score   support

     neutral       0.79      0.32      0.46        59
   confident       0.00      0.00      0.00         0
     anxious       0.82      0.25      0.39        55
       happy       0.31      0.92      0.46        59
         sad       0.46      0.46      0.46        41
       angry       0.84      0.37      0.52        43
   surprised       0.90      0.41      0.56        22
   uncertain       1.00      0.57      0.73        21

    accuracy                           0.48       300
   macro avg       0.64      0.41      0.45       300
weighted avg       0.69      0.48      0.48       300

  [BELOW THRESHOLD]  0.4467

[Text] Loading xlm_roberta_sentiment.onnx


...

0000.parquet:   0%|          | 0.00/291k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

[Text] 300 omilab/hebrew_sentiment (test) samples
  [Text] 200/300  (28s elapsed)

  Text — MAE: 0.0458  (lower is better; <0.15 = PASS)
  [PASS]  MAE=0.0458

[Fusion] Loading cross_modal_transformer.onnx
[Fusion] 400 synthetic samples

  Fusion — Macro F1: 0.9825  (threshold: 0.5)
              precision    recall  f1-score   support

     neutral       0.98      0.96      0.97        50
   confident       1.00      0.98      0.99        50
     anxious       0.94      0.98      0.96        50
       happy       0.98      1.00      0.99        50
         sad       0.98      0.96      0.97        50
       angry       1.00      1.00      1.00        50
   surprised       0.98      1.00      0.99        50
   uncertain       1.00      0.98      0.99        50

    accuracy                           0.98       400
   macro avg       0.98      0.98      0.98       400
weighted avg       0.98      0.98      0.98       400

  [PASS]  0.9825

================================================

/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:147: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
